In [1]:
from rl_trading_lab.environment.trading_env import TradingEnv, Action

In [2]:
import logging

# Configure logging
logging.basicConfig(
  level=logging.DEBUG,
  format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

- Create a small pandas DataFrame suitable for initializing TradingEnv.
- Columns: timestamp, open, high, low, close, volume.
- Generate 200 rows of plausible OHLCV data with a datetime index.

In [3]:
import pandas as pd
import numpy as np

np.random.seed(42)

periods = 200
dates = pd.date_range(start="2022-01-01", periods=periods, freq="D")
price = 100 + np.cumsum(np.random.normal(0, 1, size=periods)).round(2)
vol = np.random.randint(1000, 5000, size=periods)

df = pd.DataFrame({
    "timestamp": dates,
    "open": price + np.random.normal(0, 0.5, size=periods),
    "high": price + np.abs(np.random.normal(0.8, 0.6, size=periods)),
    "low": price - np.abs(np.random.normal(0.8, 0.6, size=periods)),
    "close": price,
    "volume": vol
})

df["open"] = df["open"].round(2)
df["high"] = df[["open", "close"]].max(axis=1).where(df["high"] < df[["open", "close"]].max(axis=1), df["high"]).round(
    2)
df["low"] = df[["open", "close"]].min(axis=1).where(df["low"] > df[["open", "close"]].min(axis=1), df["low"]).round(2)
df["close"] = df["close"].round(2)

df


,timestamp,open,high,low,close,volume
0,2022-01-01,100.06,101.07,100.06,100.50,2409
1,2022-01-02,100.01,100.52,97.88,100.36,1784
2,2022-01-03,100.80,101.92,99.25,101.01,4175
3,2022-01-04,102.47,102.86,101.74,102.53,4464
4,2022-01-05,101.53,103.11,101.53,102.30,4882
...,...,...,...,...,...,...
195,2022-07-15,93.08,94.22,93.08,93.66,2440
196,2022-07-16,92.76,92.97,92.02,92.78,2191
197,2022-07-17,92.48,94.15,92.24,92.93,4913
198,2022-07-18,93.05,93.93,92.11,92.99,2066


In [4]:
env = TradingEnv(df, lookback_window=0, randomize_start=False)

2025-10-29 08:56:41,989 - rl_trading_lab.environment.trading_env - INFO - TradingEnv initialized: randomize_start=False, hold_closes_position=False, min_episode_length=100, reward_type=sharpe, data_length=200


In [5]:
obs = env.reset()
obs

2025-10-29 08:56:42,030 - rl_trading_lab.environment.trading_env - DEBUG - Episode reset: fixed start at step 0


(array([1.0006e+02, 1.0107e+02, 1.0006e+02, 1.0050e+02, 2.4090e+03,
        0.0000e+00, 0.0000e+00, 1.0000e+00], dtype=float32),
 {'step': 0,
  'cash': 10000,
  'portfolio_value': 10000,
  'position': 0.0,
  'total_return': 0.0,
  'num_trades': 0})

In [6]:
obs, reward, terminated, truncated, info = env.step(action=Action.BUY)

2025-10-29 08:56:42,059 - rl_trading_lab.environment.portfolio - DEBUG - Opening position: signal=1.0, price=100.50
2025-10-29 08:56:42,060 - rl_trading_lab.environment.portfolio - DEBUG - Trade #1: LONG 94.5274 @ $100.55 (cash flow: -$9514.25, remaining: $485.75)
2025-10-29 08:56:42,060 - rl_trading_lab.environment.portfolio - DEBUG - Position opened: size=94.5274


In [7]:
obs

array([ 1.0001000e+02,  1.0052000e+02,  9.7879997e+01,  1.0036000e+02,
        1.7840000e+03,  9.4527367e+01, -1.8920888e-03,  9.9725115e-01],
      dtype=float32)

In [8]:
reward

np.float64(-0.002748858084577114)

In [9]:
terminated

False

In [10]:
info

{'step': 1,
 'cash': np.float64(485.74524999999994),
 'portfolio_value': np.float64(9972.511419154229),
 'position': np.float64(94.5273631840796),
 'total_return': np.float64(-0.002748858084577114),
 'num_trades': 1}

In [11]:
info['position'] * obs[3]

np.float64(9486.766226849153)

In [12]:
94.5273631840796 * 100.55 + 485.74524999999994

9990.471618159203

In [13]:
(9990.471618159203 / 10_000 - 1)

-0.0009528381840796518

In [14]:
env.get_trade_history()

[]

In [15]:
(9972.511419154229 / 10_000 - 1)

-0.0027488580845771438

In [16]:
obs, reward, terminated, truncated, info = env.step(action=Action.SELL)

2025-10-29 08:58:01,945 - rl_trading_lab.environment.portfolio - DEBUG - Closing position: current=94.5274, signal=-1.0
2025-10-29 08:58:01,946 - rl_trading_lab.environment.portfolio - DEBUG - Position closed: P&L=$-22.73, Commission=$9.48, Net=$-32.21 (cash flow: +$9482.02, -$9.48, balance: $9958.29)


In [17]:
env.get_trade_history()

[{'trade_id': 1,
  'open_step': 0,
  'open_timestamp': Timestamp('2022-01-01 00:00:00'),
  'side': 'LONG',
  'entry_price': np.float64(100.55024999999999),
  'position_size': np.float64(94.5273631840796),
  'entry_commission': np.float64(9.50475),
  'close_step': 1,
  'close_timestamp': Timestamp('2022-01-02 00:00:00'),
  'exit_price': np.float64(100.30982),
  'pnl': np.float64(-22.727213930347244),
  'exit_commission': np.float64(9.482022786069653),
  'net_pnl': np.float64(-32.209236716416896),
  'return_pct': np.float64(-0.003388751594351971),
  'hold_bars': 1}]

In [18]:
obs

array([1.008000e+02, 1.019200e+02, 9.925000e+01, 1.010100e+02,
       4.175000e+03, 0.000000e+00, 0.000000e+00, 9.958286e-01],
      dtype=float32)

In [19]:
info

{'step': 2,
 'cash': np.float64(9958.286013283583),
 'portfolio_value': np.float64(9958.286013283583),
 'position': 0.0,
 'total_return': np.float64(-0.004171398671641691),
 'num_trades': 1,
 'sharpe': np.float64(-50.1212397096836),
 'max_drawdown': np.float64(0.0014264617279177037)}

In [20]:
from stable_baselines3.common.vec_env import VecFrameStack
vec_env = VecFrameStack(env, n_stack=4)

2025-10-29 09:05:07,326 - matplotlib - DEBUG - matplotlib data path: /Users/mohamedali/trading_project/rl-trading-lab/.venv/lib/python3.12/site-packages/matplotlib/mpl-data
2025-10-29 09:05:07,330 - matplotlib - DEBUG - CONFIGDIR=/Users/mohamedali/.matplotlib
2025-10-29 09:05:07,337 - matplotlib - DEBUG - interactive is False
2025-10-29 09:05:07,338 - matplotlib - DEBUG - platform is darwin
2025-10-29 09:05:07,356 - matplotlib - DEBUG - CACHEDIR=/Users/mohamedali/.matplotlib
2025-10-29 09:05:07,358 - matplotlib.font_manager - DEBUG - Using fontManager instance from /Users/mohamedali/.matplotlib/fontlist-v390.json


AttributeError: 'TradingEnv' object has no attribute 'num_envs'

In [21]:
from stable_baselines3.common.monitor import Monitor
m_vec = Monitor(env, filename='monitor.csv')

In [22]:
m_vec.reset()

2025-10-29 09:07:00,659 - rl_trading_lab.environment.trading_env - DEBUG - Episode reset: fixed start at step 0


(array([1.0006e+02, 1.0107e+02, 1.0006e+02, 1.0050e+02, 2.4090e+03,
        0.0000e+00, 0.0000e+00, 1.0000e+00], dtype=float32),
 {'step': 0,
  'cash': 10000,
  'portfolio_value': 10000,
  'position': 0.0,
  'total_return': 0.0,
  'num_trades': 0})

In [24]:
for _ in range(100):
    action = m_vec.action_space.sample()
    m_vec.step(action)

2025-10-29 09:07:44,659 - rl_trading_lab.environment.portfolio - DEBUG - Opening position: signal=1.0, price=100.36
2025-10-29 09:07:44,659 - rl_trading_lab.environment.portfolio - DEBUG - Trade #1: LONG 94.6592 @ $100.41 (cash flow: -$9514.25, remaining: $485.75)
2025-10-29 09:07:44,660 - rl_trading_lab.environment.portfolio - DEBUG - Position opened: size=94.6592
2025-10-29 09:07:44,660 - rl_trading_lab.environment.portfolio - DEBUG - Closing position: current=94.6592, signal=-1.0
2025-10-29 09:07:44,661 - rl_trading_lab.environment.portfolio - DEBUG - Position closed: P&L=$52.00, Commission=$9.56, Net=$42.44 (cash flow: +$9556.75, -$9.56, balance: $10032.94)
2025-10-29 09:07:44,662 - rl_trading_lab.environment.portfolio - DEBUG - Opening position: signal=1.0, price=102.53
2025-10-29 09:07:44,662 - rl_trading_lab.environment.portfolio - DEBUG - Trade #2: LONG 92.9610 @ $102.58 (cash flow: -$9545.59, remaining: $487.35)
2025-10-29 09:07:44,663 - rl_trading_lab.environment.portfolio - 

In [25]:
m_vec.get_episode_lengths()

[]

In [26]:
m_vec.get_episode_rewards()

[]

In [27]:
m_vec.get_total_steps()

100